
# Benchmark LLM — AO-BTP Copilot

<div style="color:#7f8c8d;font-size:14px">
Évaluation comparative des fournisseurs de modèles candidats pour l'ago RAG du
projet AO-BTP Copilot. Le benchmark est <b>grounded</b> : il alimente chaque modèle
avec le <b>même</b> contexte (extraits du Recueil ARCOP 2024) et vérifie que la
réponse <em>suit</em> ce contexte, cite les bons articles, et produit un français
opérationnel propre au secteur des marchés publics togolais.
</div>

## Les 7 critères

| # | Critère | Nature | Pondération |
|---|---------|--------|-------------|
| 1 | Fidélité au contexte (grounding) | Auto (pièges) + juge | élevée |
| 2 | Citations d'articles exactes | Auto (texte présent) | élevée |
| 3 | Qualité du français / jargon | Juge LLM | moyenne |
| 4 | Latence (temps de réponse) | Mesurée | moyenne |
| 5 | Coût réel par requête | Calcul tokens × tarif | moyenne |
| 6 | Tenue des quotas (RPM/RPD) | Modélisé | moyenne |
| 7 | Robustesse / format / honnêteté | Auto (réponses hors contexte) + juge | élevée |

> **Protocole de scoring** : un juge LLM (un modèle **non concurrent** du candidat,
> par défaut Gemini) note les aspects qualitatifs ; un échantillon manuel est
> réservé pour validation par l'utilisateur.

> ⚠️ **Aucune clé n'est présente dans ce dépôt.**
> * Le fichier `.env` est créer pour utiliser les clés d'API en situation rée;, lancer `1.2` et la plupart
>   des sections suivantes tourneront en **mode trace** (aucun appel réseau).
> * Copie de  `.env.example` vers `.env` puis ajouter `GEMINI_API_KEY` et/ou
>   `GROQ_API_KEY` pour passer en **mode réel**.



## 1. Import du noyau et configuration

Le noyau `src/llm_benchmark.py` contient toute la logique (clients, prompts,
scoring) ; le notebook ne fait que piloter.


In [1]:
# %% [markdown]
import os, sys, json, re, time, random, csv
from pathlib import Path
from datetime import datetime

# Racine du projet : on est soit dans notebooks/, soit déjà à la racine
_cwd = Path.cwd()
PROJ = _cwd if (_cwd / ".env").exists() else _cwd.parent

sys.path.insert(0, str(PROJ / "src"))

from dotenv import load_dotenv
load_dotenv(PROJ / ".env")  # charge les clés API depuis .env (racine, jamais commité)

import llm_benchmark as lb

# --- Paramètres ---------------------------------------------------
MAX_OUT = 1500                 # tokens max de sortie par réponse
SAVE_DIFF = True               # exporte résultats bruts en CSV
SEED = 42                      # reproductibilité (mode trace)
random.seed(SEED)

# Liste des modèles testés (ordres stables pour les tableaux)
FOURNISSEURS = ["gemini", "groq", "ollama"]

# Clés présentes ? (le `.env` est hors Git)
# gemini_ok / groq_ok → 1.3 gère l'auto-mocking du mode trace
KEYS = {
    "gemini": bool(os.environ.get("GEMINI_API_KEY")),
    "groq":   bool(os.environ.get("GROQ_API_KEY")),
}

print("Modèles candidats :", ", ".join(FOURNISSEURS))
print("Clés présentes :", {k: bool(v) for k, v in KEYS.items()})
print("Fichier .env :", "présent" if Path(".env").exists() else "ABSENT (mode trace)")

Modèles candidats : gemini, groq, ollama
Clés présentes : {'gemini': True, 'groq': True}
Fichier .env : ABSENT (mode trace)



### 1.1 Modèles disponibles

**Forcés par l'utilisateur (à challenger)** :
* `gemini-3.5-flash` — Gemini 3.5 Flash (gratuit, contexte 1M ; **quota free tier réel = 20 req/jour**, vérifié le 17/08/2026).
* `groq-gpt-oss-120b` — Groq + openai/gpt-oss-120b (gratuit, API
  OpenAI-compatible).
* `ollama` — local (à activer manuellement via la section **9**).

**Params clés (temps-out réseau, graphes de retry) modifiables** si besoin.


In [ ]:
# Récap du protocole
print("\n==== PROTOCOLE ====")
for p in FOURNISSEURS:
    print(f"  - {p}: clé {'présente' if KEYS.get(p) else 'ABSENTE -> mode trace'}")

# --- Choix du juge LLM (test RÉEL de disponibilité) --------------------------
# Attention : avoir une clé ne suffit pas. Gemini peut être indisponible, par
# exemple si son quota journalier (20 req/jour en free tier) est déjà épuisé.
# On teste donc un MICRO-appel : le premier candidat qui répond devient juge.
# Ordre : Gemini d'abord (non concurrent de Groq), puis Groq (auto-évaluation
# pour Groq, acceptable à défaut, à valider par l'échantillon manuel, §5).
def _tester_dispo_juge(provider):
    d = lb.call_model(provider,
                      "Réponds UNIQUEMENT par OUI.",
                      "Micro-test de disponibilité du juge.",
                      max_output_tokens=5, retry_429=0)
    return d["ok"], d.get("erreur")

JUGE = None
for _cand in ("gemini", "groq"):
    if not KEYS.get(_cand):
        continue
    _ok, _err = _tester_dispo_juge(_cand)
    if _ok:
        JUGE = _cand
        print(f"\nJuge LLM : {_cand} (micro-appel réussi)")
        break
    print(f"  juge {_cand} indisponible : {(_err or '')[:90]}")

if not JUGE:
    print("\nJuge LLM : AUCUN (échantillon manuel requis, §5)")


### 1.2 Jeu d'évaluation

`data/eval/eval_questions.json` : 7 questions ancrées sur des articles réels du
Recueil (Art. 12, 27, 28, 110-111, 113), 1 piège de grounding et 1 question à
réponse absente du contexte (anti-hallucination).


In [ ]:
# Chargement du jeu
questions = lb.load_eval_set()
print(f"{len(questions)} questions chargées :")
for q in questions:
    piege = "piège" if q.get("piege_grounding") else ""
    abs_ = "info absente" if q.get("categorie") == "info_absente" else ""
    print(f"  - {q['id']}: {q['question'][:70]}...{piege}{abs_}")

7 questions chargées :
  - proc-01: Quelle est la procédure par laquelle l'autorité contractante choisit l...
  - delai-02: Quel est le délai minimal de réception des offres pour un PPP dont la ...
  - garantie-03: Quel est le taux de la garantie de bonne exécution et dans quel délai ...
  - piege-04: Selon le texte fourni, la garantie de bonne exécution est-elle exigée ...
  - eligible-05: Quelles sont les conditions de participation que les autorités contrac...
  - absent-06: Selon le texte fourni, quel est le délai de retrait d'un dossier d'app... 🚫 info absente
  - presta-intel-07: Pour quel type de marché la garantie de bonne exécution et la retenue ...



### 1.3 Cohérence du protocole de scoring

Le juge est un **LLM non concurrent** (Gemini juge Groq / Ollama ; Groq juge
Gemini) pour éviter l'auto-évaluation. Toute sortie d'un modèle passe d'abord
par le noyau (`call_model`) qui ne touche jamais au réseau sans clé.


In [4]:
# Mode trace : aucune clé ? Aucun appel réseau ne sera fait.
# On construit un "mock" local qui imite une réponse plausible (dérivée du label)
# pour vérifier le pipeline de scoring — explicitement marqué comme simulé.
_MOCK = {}
for q in questions:
    if q.get("categorie") == "info_absente":
        _MOCK[q["id"]] = "Selon le texte fourni, aucune information relative au délai de retrait n'est présentée. Je ne peux pas répondre précisément."
    elif q.get("piège_grounding"):
        _MOCK[q["id"]] = "Non, la garantie n'est pas exigée pour les travaux dans ce texte ; elle est uniquement requise pour les fournitures, au taux de 2 %."
    else:
        _MOCK[q["id"]] = "D'après le contexte (Article %s), ..." % (q.get("citation_attendue") or [""])[0].replace("Article ", "")

def _real_call(provider):
    def fn(system, user):
        res = lb.call_model(provider, system, user, max_output_tokens=MAX_OUT)
        return res
    return fn

def _mock_call(provider):
    def fn(system, user):
        # extraction de l'id de la question : le user contient "Question : <id>"
        m = re.search(r"Question\s*:\s*(\S+)", user)
        qid = m.group(1) if m else "unknown"
        txt = _MOCK.get(qid, "(mode trace) réponse simulée.")
        return {"text": txt, "usage_in": 200, "usage_out": 180,
                "ok": True, "erreur": None, "duree_s": 0.1, "simulé": True}
    return fn

def get_caller(provider):
    # Renvoie (fn, simulate) : fn(system, user) -> dict, simulate=True si mock.
    if KEYS.get(provider):
        return _real_call(provider), False
    return _mock_call(provider), True

print("Mode trace actif pour :",
      [p for p in FOURNISSEURS if not KEYS.get(p)])
print("Prêt : run de la cellule 4 (benchmark).")

Mode trace actif pour : ['ollama']
Prêt : run de la cellule 4 (benchmark).



## 2. Benchmark — toutes questions × tous modèles

Boucle de base : pour chaque question, chaque modèle reçoit le **même contexte**,
puis on collecte (texte, durée, usage).


In [5]:
# Boucle principale : collecte des réponses
# structure : results[provider][qid] = résultat complet
results = {}

for p in FOURNISSEURS:
    caller, simulated = get_caller(p)
    results[p] = {}
    for q in questions:
        sys_p, usr_p = lb.build_prompt(q, "")
        res = caller(sys_p, usr_p)
        res["simulé"] = simulated
        results[p][q["id"]] = res

# Synthèse brute consolide
print("\nRésultats bruts (durée, tokens, ok, simulé) :")
for p in FOURNISSEURS:
    print(f"\n--- {p} ---")
    for q in questions:
        r = results[p][q["id"]]
        print(f"  {q['id']}: ok={r['ok']} duree={r.get('duree_s',0):.2f}s "
              f"tu={r['usage_in']} to={r['usage_out']} "
              + ("(SIMULÉ)" if r.get("simulé") else "")
              + ("" if r.get("ok") else f" | ERREUR: {r.get('erreur')}"))

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.



Résultats bruts (durée, tokens, ok, simulé) :

--- gemini ---
  proc-01: ok=False duree=0.00s tu=0 to=0  | ERREUR: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 23.465420528s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'Generat


### 2.1 Résultats bruts (observables)

Chaque ligne : réponse, durée (s), tokens in/out, échec éventuel. La durée est
mesurée par le noyau. En **mode trace**, les durées sont simulées (0,1 s).


In [6]:
# Aperçu des réponses (échantillon manuel, 2 premières questions × 3 modèles)
print("\nAperçu des réponses (mode " +
      ("SIMULÉ" if not any(KEYS.get(p) for p in FOURNISSEURS) else "RÉEL") +
      ") :")
for p in ["gemini", "groq", "ollama"]:
    print(f"\n### {p} — {questions[0]['id']} : {questions[0]['question'][:60]}...")
    print(results[p][questions[0]["id"]]["text"][:700])


Aperçu des réponses (mode RÉEL) :

### gemini — proc-01 : Quelle est la procédure par laquelle l'autorité contractante...


### groq — proc-01 : Quelle est la procédure par laquelle l'autorité contractante...
La procédure décrite est **l’appel d’offres ouvert en une étape** : c’est celle par laquelle l’autorité contractante sélectionne l’offre économiquement la plus avantageuse, sans aucune négociation, sur la base de critères objectifs préalablement publiés.

Oui, dans ce cadre un opérateur économique peut **soumettre un dossier de demande de pré‑qualification** lorsque l’appel d’offres est précédé d’une phase de pré‑qualification (cf. le texte : « L’appel d’offres ouvert en une étape peut être précédé d’une pré‑qualification… Tout opérateur économique intéressé peut… soumettre un dossier de demande de pré‑qualification »).  

*Note : le texte fourni ne précise pas de numéro d’article, il s’agit

### ollama — proc-01 : Quelle est la procédure par laquelle l'autorité contractante...
(


## 3. Scoring automatique (observable, sans juge)

* **Grounding piège** : piège-04 doit être suivi (non pour travaux, 2 %).
* **Honnêteté** : absent-06 doit donner "pas dans le contexte" et non inventer.
* **Citations** : au moins une citation attendue doit apparaître.
* **Faits** : proportion des éléments attendus présents (grounding positif).


In [7]:
# Scoring automatique sur les réponses collectées
# (grounding piège, honnêteté info absente, citations, faits)
auto_scores = {}
for p in FOURNISSEURS:
    auto_scores[p] = {}
    for q in questions:
        rep = results[p][q["id"]]["text"]
        c = q.get("categorie")
        s = {"grounding_piege": 1.0 if c != "piege_grounding" else
                                  lb.score_piege_grounding(rep, q),
             "info_absente":   1.0 if c != "info_absente" else
                                  lb.score_info_absente(rep, q),
             "citation":       lb.score_citations(rep, q.get("citation_attendue", [])),
             "faits":          lb.score_faits_presents(rep, q)}
        auto_scores[p][q["id"]] = s

print("\nScoring automatique (grounding piège / honnêteté / citations / faits) :")
for p in FOURNISSEURS:
    print(f"\n--- {p} ---")
    for q in questions:
        s = auto_scores[p][q["id"]]
        print(f"  {q['id']}: piège={s['grounding_piege']:.2f} "
              f"infoA={s['info_absente']:.2f} cit={s['citation']:.2f} "
              f"faits={s['faits']:.2f}")


Scoring automatique (grounding piège / honnêteté / citations / faits) :

--- gemini ---
  proc-01: piège=1.00 infoA=1.00 cit=0.00 faits=0.00
  delai-02: piège=1.00 infoA=1.00 cit=0.00 faits=0.00
  garantie-03: piège=1.00 infoA=1.00 cit=0.00 faits=0.00
  piege-04: piège=0.00 infoA=1.00 cit=1.00 faits=0.00
  eligible-05: piège=1.00 infoA=1.00 cit=0.00 faits=0.00
  absent-06: piège=1.00 infoA=0.00 cit=1.00 faits=0.00
  presta-intel-07: piège=1.00 infoA=1.00 cit=0.00 faits=0.00

--- groq ---
  proc-01: piège=1.00 infoA=1.00 cit=0.00 faits=0.25
  delai-02: piège=1.00 infoA=1.00 cit=0.00 faits=0.67
  garantie-03: piège=1.00 infoA=1.00 cit=0.00 faits=0.40
  piege-04: piège=1.00 infoA=1.00 cit=1.00 faits=0.75
  eligible-05: piège=1.00 infoA=1.00 cit=0.00 faits=1.00
  absent-06: piège=1.00 infoA=1.00 cit=1.00 faits=0.00
  presta-intel-07: piège=1.00 infoA=1.00 cit=0.00 faits=1.00

--- ollama ---
  proc-01: piège=1.00 infoA=1.00 cit=0.00 faits=0.00
  delai-02: piège=1.00 infoA=1.00 cit=0.00 fai


## 4. Juge LLM (aspects qualitatifs)

Le juge reçoit (contexte, question, label, réponse) et note sur 5 :
**Langue**, **Conformité**, **Format**. Il répond à un format fixe que le noyau
parse automatiquement. Si le juge n'est pas dispo, score -1 → considéré non
retenu (à compléter en manuel).


In [8]:
# Juge LLM — aspects qualitatifs (Langue / Conformité / Format)
# S'il n'y a pas de juge (aucune clé), scores -1 → compléter manuellement (5).
judge_scores = {}
if JUGE:
    print("\nÉvaluation par le juge LLM :", JUGE)
    for p in FOURNISSEURS:
        judge_scores[p] = {}
        for q in questions:
            rep = results[p][q["id"]]["text"]
            j = lb.score_juge_llm(JUGE, q, rep)
            judge_scores[p][q["id"]] = j
            print(f"  {p} {q['id']}: langue={j['langue']} conformité={j['conformite']} "
                  f"format={j['format']} ok={j['ok']}")
else:
    print("\nAucune clé -> juge LLM indisponible. Scores -1 "
          "(complétez en 5 / 6). L'échantillon manuel couvrira ce besoin.")


Évaluation par le juge LLM : gemini


  gemini proc-01: langue=-1 conformité=-1 format=-1 ok=False


  gemini delai-02: langue=-1 conformité=-1 format=-1 ok=False


  gemini garantie-03: langue=-1 conformité=-1 format=-1 ok=False


  gemini piege-04: langue=-1 conformité=-1 format=-1 ok=False


...

  ollama garantie-03: langue=-1 conformité=-1 format=-1 ok=False


  ollama piege-04: langue=-1 conformité=-1 format=-1 ok=False


  ollama eligible-05: langue=-1 conformité=-1 format=-1 ok=False


  ollama absent-06: langue=-1 conformité=-1 format=-1 ok=False


  ollama presta-intel-07: langue=-1 conformité=-1 format=-1 ok=False



## 5. Échantillon manuel

Afin de valider les auto-scores, sélectionner **~3 réponses par modèle** à
revoir à la main. L'utilisateur attribue un score 0-5 sur les 4 dimensions.
Cela sert de **contre-échantillon** à la note finale.


In [9]:
# Échantillon manuel — l'utilisateur remplit MANUELS (dict) pour ~3 questions
# dans chaque modèle. La cellule ci-dessous est JNE (code à décommenter),
# ou bien l'on s'appuie sur la cellule 5 (aperçu) pour noter à la main.
#
# manuels = {
#   "gemini": {"proc-01": {"langue":5, "conformite":4, "format":5},
#              "piege-04": {...}, "absent-06": {...}},
#   "groq": {...}, "ollama": {...},
# }
print("\nManagement manuel : référez-vous à la cellule 8 (aperçu) et remplissez "
      "le dict `manuels` ci-dessous pour ~3 questions. (voir cellule 'note')")


Management manuel : référez-vous à la cellule 8 (aperçu) et remplissez le dict `manuels` ci-dessous pour ~3 questions. (voir cellule 'note')



## 6. Synthèse pondérée & export

La note finale (sur 10) est la moyenne pondérée des critères. Un **CSV** des
résultats est écrit dans `data/processed/` (hors Git).


In [10]:
# ---- Agrégation par fournisseur ----
# 1) critères auto groupés par question->fournisseur (moyenne)
# 2) juge LLM (si dispo) ou -1
# 3) latence moy., coût estimé (tokens × tarif), quotas (simplifié), robustesse
import statistics

def _moy(liste):
    vals = [v for v in liste if v is not None]
    return statistics.mean(vals) if vals else -1

synthese = {}
for p in FOURNISSEURS:
    lst = []
    lst.append(("grounding", _moy([auto_scores[p][q["id"]]["grounding_piege"]
                                     for q in questions])))
    lst.append(("citations", _moy([auto_scores[p][q["id"]]["citation"]
                                     for q in questions])))
    lst.append(("robustesse", _moy([auto_scores[p][q["id"]]["info_absente"]
                                      for q in questions])))
    # langue / conformité / format : du juge (ou -1)
    j = judge_scores.get(p, {})
    lst.append(("langue", _moy([j.get(q["id"], {}).get("langue", -1)
                                  for q in questions])
                if j else -1))
    lst.append(("conformite", _moy([j.get(q["id"], {}).get("conformite", -1)
                                      for q in questions])
                if j else -1))
    # latence : moyenne des durées mesurées
    lat = _moy([results[p][q["id"]].get("duree_s") for q in questions])
    lst.append(("latence", lat))
    # coût estimé (USD / 1M tokens) — grille gratuite des challengers
    tokens_in = sum(results[p][q["id"]]["usage_in"] for q in questions)
    tokens_out = sum(results[p][q["id"]]["usage_out"] for q in questions)
    mod = lb.MODELS_CATALOG.get(("gemini-3.5-flash" if p == "gemini"
                                 else ("groq-gpt-oss-120b"
                                       if p == "groq" else "ollama")), {})
    cout = (tokens_in * mod.get("prix_in_par_million", 0) +
            tokens_out * mod.get("prix_out_par_million", 0)) / 1_000_000
    lst.append(("cout", cout))
    # quotas : grille gratuite ≈ proportion quota utilisés sur 50 AO/jour
    rpm = mod.get("quota_rpm", 0)
    rpd = mod.get("quota_rpd", 0)
    lst.append(("quotas", _moy([min(1.0, 50 / rpd) for _ in questions]) if rpd else -1))
    synthese[p] = dict(lst)

print("\n=== SYNTHÈSE (par fournisseur) ===")
for p, d in synthese.items():
    print(f"\n{p}:")
    for k, v in d.items():
        print(f"  {k:14} : {v}")

# -------- Note finale pondérée ----------
print("\nNote finale (pondérée, /10) :")
for p in FOURNISSEURS:
    d = synthese[p]
    # linearise et borne les critères transformés (0-1)
    lat  = min(1.0, 60.0 / d["latence"]) if d["latence"] and d["latence"] > 0 else -1
    cout = max(0.0, 1.0 - d["cout"])
    note = lb.calcule_note_finale({
        "grounding": d["grounding"], "citations": d["citations"],
        "robustesse": d["robustesse"], "langue": (d["langue"] / 5 if d["langue"] >= 0 else -1),
        "latence": lat, "quotas": d["quotas"], "cout": cout})
    print(f"  {p:12} : {round(note, 2)} / 10")


=== SYNTHÈSE (par fournisseur) ===

gemini:
  grounding      : 0.8571428571428571
  citations      : 0.2857142857142857
  robustesse     : 0.8571428571428571
  langue         : -1
  conformite     : -1
  latence        : -1
  cout           : 0.0
  quotas         : 0.03333333333333333

groq:
  grounding      : 1.0
  citations      : 0.2857142857142857
  robustesse     : 1.0
  langue         : -1
  conformite     : -1
  latence        : 3.0396588714289203
  cout           : 0.0
  quotas         : 0.003472222222222222

ollama:
  grounding      : 0.8571428571428571
  citations      : 0.2857142857142857
  robustesse     : 0.8571428571428571
  langue         : -1
  conformite     : -1
  latence        : 0.1
  cout           : 0.0
  quotas         : -1

Note finale (pondérée, /10) :
  gemini       : 2.96 / 10
  groq         : 5.57 / 10
  ollama       : 3.93 / 10


In [11]:
# Export CSV des résultats (hors Git)
out_dir = Path.cwd().parent / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
fname = out_dir / f"benchmark_results_{ts}.csv"
with open(fname, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["fournisseur", "question", "ok", "simulé", "durée_s",
                "tokens_in", "tokens_out", "texte"])
    for p in FOURNISSEURS:
        for q in questions:
            r = results[p][q["id"]]
            w.writerow([p, q["id"], r["ok"], r.get("simulé", False),
                        round(r.get("duree_s", 0), 3),
                        r["usage_in"], r["usage_out"],
                        r["text"]])
print("Écrit :", fname)

Écrit : C:\Users\angel\OneDrive\Desktop\AO-BTP Copilot\data\processed\benchmark_results_20260817_173043.csv



## 7. (Décommenter pour) Ollama local

Ollama n'a pas besoin de clé : il tourne en local. Pour activer :
1. Installez Ollama (https://ollama.com) et un modèle, ex. `ollama pull llama3.2:1b`.
2. Décommentez la cellule ci-dessous et indiquez le modèle.
3. Relancez le notebook — Ollama est alors ajouté comme candidat.

> ⚠️ Modèle retenu (18/08/2026) : **`llama3.2:1b`** (léger, 1.3 GB) — remplace
> `qwen3.6:27b` retiré pour libérer le disque.


In [12]:
# ========== OLLAMA LOCAL (à activer manuellement) ==========
# Cette cellule est INACTIVE (marquée out [*]) par défaut.
# Pour activer : installez un modèle puis dé-commentez les lignes actives.
# 1) Ollama doit tourner (binaire installé + `ollama serve`)
# 2) Téléchargez un modèle : `ollama pull llama3.2:1b`
# 3) ICI : remplacez le modèle et relancez

OLLAMA_MODELE = "llama3.2:1b"      # <-- modèle à adapter
OLLAMA_HOST = "http://localhost:11434"

# Pour activer, supprimez le # des deux lignes ci-dessous :
# FOURNISSEURS.append("ollama")  # ajoute le fournisseur à la boucle
# OLLAMA_CLIENT = lb.build_ollama_client()  # client prêt à l'emploi

print("Ollama : section désactivée (data mobile), voir cellule 7 du notebook.")


Ollama : section désactivée (data mobile), voir cellule 7 du notebook.
